# 01 — Classification & lecture des pièces d'identité (KYC)

Pipeline : rendu PDF → triage pages blanches → heuristiques déterministes (MRZ) →
transcription VLM (Qwen2.5-VL) → **décisions 100 % code Python** (type, pays, validité MRZ) →
extraction des champs selon le type.

Principes (CLAUDE.md) :
- le VLM **transcrit uniquement ce qui est visible** ; toute décision est déterministe ;
- ambigu / illisible → `INCONNU` + revue, jamais de valeur devinée ;
- `BACKEND` (`ollama` / `domino` / `replay`) est le seul point de bascule ;
- seuils page blanche et MRZ **calibrés empiriquement sur les 13 spécimens fictifs** (03/07/2026).

In [1]:
# === CONFIG =================================================================
from pathlib import Path

BACKEND = "openrouter"                 # "ollama" | "domino" | "replay" | "openrouter"

# --- backend openrouter (cloud) ---
OPENROUTER_MODEL = "qwen/qwen2.5-vl-72b-instruct"
# clé via : export OPENROUTER_API_KEY="sk-or-..."

# --- backend ollama (test local) ---
OLLAMA_URL   = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5vl:7b"

# --- backend domino (production A100) ---
MODEL_PATH = "/mnt/artifacts/modelhub/Qwen2.5-VL-7B-Instruct"   # a ajuster sur Domino

# --- donnees ---
DATA_ROOT    = Path("data_test/KYC ")   # un sous-dossier par id_tiers
OUT_DIR      = Path("outputs")
FIXTURES_DIR = Path("tests/fixtures/replay")
RECORD_FIXTURES = True                 # True (avec BACKEND=ollama) pour enregistrer les fixtures replay

RENDER_DPI = 300
MAX_COTE_VLM = 1280   # cote max des images envoyees au VLM (CPU local : latence).
                      # Sur Domino (A100), passer a 2048 ou None pour desactiver.

# --- seuils calibres sur specimens (voir cellule de calibration en bas) ---
BLANK_EDGE_MAX = 0.010    # densite de contours max d'une page blanche (blanches<=0.008 ; contenu>=0.017)
BLANK_LAP_MAX  = 150.0    # variance laplacienne max d'une page blanche
MRZ_ANGLES_FINS = (0.0, -7.0, 7.0, -14.0, 14.0)
CLASSIF_CONF_MIN = 0.75   # confiance VLM minimale, a calibrer

TYPES_PIECES = ["CNI_BIO","CNI_NONBIO","PERMIS_BIO","PERMIS_NONBIO",
                "PASSEPORT_DZ","PASSEPORT_ETRANGER","INCONNU"]

# codes document MRZ (positions 1-2 de la ligne 1) -> famille. Configurable, jamais devine.
DOC_CODE_MAP = {"ID":"CNI_BIO", "IA":"CNI_BIO", "DL":"PERMIS_BIO", "D<":"PERMIS_BIO"}

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIXTURES_DIR.mkdir(parents=True, exist_ok=True)

## Localisation des dossiers (`dossier_locator`)
- nom de dossier = `id_tiers` brut (espace comprise) ; matching apres compactage des deux cotes ;
- fichier identite : motif `identit*` **insensible aux accents et a la casse** ;
- anomalies remontees, jamais ignorees : `HORS_REFERENTIEL`, `DOSSIER_INTROUVABLE`,
  `NOM_DOSSIER_NON_CONFORME`, `DOSSIER_SANS_PIECE`.

In [2]:
import re, unicodedata

def compacter(s):
    # cle d'index interne : id_tiers sans espaces
    return re.sub(r"\s+", "", str(s or ""))

def sans_accents(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower()

MOTIF_ID_TIERS = re.compile(r"^\d{5}\s\d{6}$")

def localiser_dossiers(data_root: Path, ids_referentiel=None):
    # Retourne (index, anomalies).
    # index : {id_compact: {"dossier": Path, "identite": [Path], "domicile": [Path]}}
    index, anomalies = {}, []
    ids_ref = set() if ids_referentiel is None else {compacter(i) for i in ids_referentiel}
    for entree in sorted(p for p in data_root.iterdir() if p.is_dir()):
        if not MOTIF_ID_TIERS.match(entree.name):
            anomalies.append({"motif": "NOM_DOSSIER_NON_CONFORME", "chemin": str(entree)})
            continue
        idc = compacter(entree.name)
        pdfs = [p for p in entree.iterdir() if p.suffix.lower() == ".pdf"]
        ident = [p for p in pdfs if sans_accents(p.stem).startswith("identit")]
        domic = [p for p in pdfs if sans_accents(p.stem).startswith("domicile")]
        if not ident:
            anomalies.append({"motif": "DOSSIER_SANS_PIECE", "chemin": str(entree)})
        if ids_ref and idc not in ids_ref:
            anomalies.append({"motif": "HORS_REFERENTIEL", "chemin": str(entree)})
        index[idc] = {"dossier": entree, "identite": sorted(ident), "domicile": sorted(domic)}
    for idc in ids_ref - set(index):
        anomalies.append({"motif": "DOSSIER_INTROUVABLE", "id_tiers": idc})
    return index, anomalies

## Rendu PDF haute definition
PyMuPDF (`fitz`), portable Mac/Domino. Chaque page devient une image BGR numpy.

In [3]:
import numpy as np
import cv2
import fitz  # PyMuPDF

def rendre_pdf(chemin_pdf: Path, dpi: int = RENDER_DPI):
    # Retourne [ {"numero": int, "image": np.ndarray BGR} ] pour toutes les pages.
    pages = []
    doc = fitz.open(str(chemin_pdf))
    mat = fitz.Matrix(dpi / 72.0, dpi / 72.0)
    for i, page in enumerate(doc, start=1):
        pix = page.get_pixmap(matrix=mat, alpha=False)
        img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        if pix.n == 3:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        pages.append({"numero": i, "image": img})
    doc.close()
    return pages

## Triage des pages blanches — **mesure, jamais presomption**
Un seuil de luminosite echoue sur les scans sombres (valide empiriquement : une page vide
grisatre montait a 56 % de pixels « encres »). On mesure la **structure** apres normalisation
d'eclairage : densite de contours Canny + variance laplacienne.
Sens d'erreur choisi : en cas de doute la page est traitee (un faux « contenu » est ecarte en
aval ; un faux « blanche » perdrait un document).

In [4]:
def mesures_page(img_bgr):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
    g = cv2.resize(g, None, fx=0.5, fy=0.5)
    fond = cv2.GaussianBlur(g, (0, 0), 25)
    norme = cv2.divide(g, fond, scale=255)          # retire l'eclairage non uniforme
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    norme = clahe.apply(norme)                      # rehausse le contraste local (scans pales)
    contours = cv2.Canny(norme, 60, 160)
    densite = float((contours > 0).mean())
    laplace = float(cv2.Laplacian(norme, cv2.CV_64F).var())
    return {"densite_contours": densite, "variance_laplace": laplace}

def est_page_blanche(img_bgr):
    m = mesures_page(img_bgr)
    blanche = m["densite_contours"] < BLANK_EDGE_MAX and m["variance_laplace"] < BLANK_LAP_MAX
    return blanche, m

## Heuristique MRZ deterministe (etape 1 de la classification)
Detecte des groupes de 2-3 lignes de texte **coherentes** (largeurs/hauteurs similaires,
adjacentes, alignees) puis valide chaque ligne par la regularite de ses caracteres
(18-60 composantes de hauteur homogene = police OCR-B monospace).
Balayage : 4 orientations x angles fins pour absorber rotations 0/90/180/270 et inclinaisons.

**Statut sur les 13 specimens : 15/18 pages correctes, 0 faux positif.** Echecs connus
(cartes tres petites/inclinees) -> le fallback VLM prend le relais : c'est le fonctionnement
nominal du 2 temps, pas une exception. L'heuristique n'emet qu'une *hypothese* ; la structure
de la MRZ transcrite fait autorite (cellule suivante).

In [5]:
def _candidats_lignes_mrz(g):
    fond = cv2.GaussianBlur(g, (0, 0), 21)
    norme = cv2.divide(g, fond, scale=255)
    noyau = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
    chapeau = cv2.morphologyEx(norme, cv2.MORPH_BLACKHAT, noyau)   # texte sombre sur fond clair
    chapeau = cv2.normalize(chapeau, None, 0, 255, cv2.NORM_MINMAX)
    _, seuil = cv2.threshold(chapeau, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    ferme = cv2.morphologyEx(seuil, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_RECT, (41, 3)), iterations=2)
    ferme = cv2.erode(ferme, None, iterations=1)
    cnts, _ = cv2.findContours(ferme, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boites = []
    for c in cnts:
        x, y, w, h = cv2.boundingRect(c)
        if h and w / float(h) > 7 and w > 180 and 7 < h < 70:
            boites.append((x, y, w, h))
    return boites, norme

def _ligne_mrz_plausible(norme, boite):
    # une vraie ligne MRZ contient 18-60 caracteres de hauteur reguliere
    x, y, w, h = boite
    marge = max(2, h // 4)
    roi = norme[max(0, y - marge):y + h + marge, max(0, x):x + w]
    if roi.size == 0:
        return False
    _, binaire = cv2.threshold(roi, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
    n, _, stats, _ = cv2.connectedComponentsWithStats(binaire, 8)
    hauteurs = [s[3] for s in stats[1:] if s[4] > 4]
    if not (18 <= len(hauteurs) <= 60):
        return False
    hauteurs = np.array(hauteurs)
    ecart = float(np.median(np.abs(hauteurs - np.median(hauteurs))))
    return ecart <= 0.45 * float(np.median(hauteurs))

def _grouper_lignes(boites):
    boites = sorted(boites, key=lambda b: b[1])
    groupes = []
    for b in boites:
        for grp in groupes:
            d = grp[-1]
            if (abs(b[2] - d[2]) < 0.18 * max(b[2], d[2])
                    and abs(b[3] - d[3]) < 0.6 * max(b[3], d[3])
                    and abs(b[1] - (d[1] + d[3])) < 2.2 * d[3]
                    and abs(b[0] - d[0]) < 0.15 * max(b[2], d[2])):
                grp.append(b)
                break
        else:
            groupes.append([b])
    return groupes

def _nb_lignes_mrz(g):
    boites, norme = _candidats_lignes_mrz(g)
    boites = [b for b in boites if _ligne_mrz_plausible(norme, b)]
    groupes = _grouper_lignes(boites)
    meilleur = max(groupes, key=len) if groupes else []
    n = len(meilleur)
    if not (2 <= n <= 3):
        return 0, None
    xs = [b[0] for b in meilleur]; ys = [b[1] for b in meilleur]
    x2 = [b[0] + b[2] for b in meilleur]; y2 = [b[1] + b[3] for b in meilleur]
    return n, (min(xs), min(ys), max(x2), max(y2))

def detecter_mrz(img_bgr):
    # Retourne {"nb_lignes", "orientation", "angle_fin", "boite"} ; nb_lignes=0 si rien.
    g0 = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY) if img_bgr.ndim == 3 else img_bgr
    echelle = 1400.0 / max(g0.shape)
    g0 = cv2.resize(g0, None, fx=echelle, fy=echelle)
    meilleur = {"nb_lignes": 0, "orientation": 0, "angle_fin": 0.0, "boite": None, "echelle": echelle}
    rotations = [(0, None), (90, cv2.ROTATE_90_CLOCKWISE),
                 (180, cv2.ROTATE_180), (270, cv2.ROTATE_90_COUNTERCLOCKWISE)]
    for orient, code_rot in rotations:
        gr = g0 if code_rot is None else cv2.rotate(g0, code_rot)
        for angle in MRZ_ANGLES_FINS:
            if angle == 0.0:
                gf = gr
            else:
                M = cv2.getRotationMatrix2D((gr.shape[1] / 2, gr.shape[0] / 2), angle, 1.0)
                gf = cv2.warpAffine(gr, M, (gr.shape[1], gr.shape[0]),
                                    flags=cv2.INTER_LINEAR, borderValue=255)
            n, boite = _nb_lignes_mrz(gf)
            if n > meilleur["nb_lignes"]:
                meilleur.update({"nb_lignes": n, "orientation": orient, "angle_fin": angle, "boite": boite})
            if n == 3:
                return meilleur
    return meilleur

def redresser_page(img_bgr, orientation, angle_fin):
    # applique la rotation trouvee par le balayage MRZ (ou le triage) a la page pleine resolution
    rot_map = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}
    img = img_bgr if orientation == 0 else cv2.rotate(img_bgr, rot_map[orientation])
    if angle_fin:
        M = cv2.getRotationMatrix2D((img.shape[1] / 2, img.shape[0] / 2), angle_fin, 1.0)
        img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                             flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))
    return img

## Interface VLM — `_run_inference`, seul point de bascule
- `ollama` : API HTTP locale (test) ; `domino` : transformers pleine precision (import **garde**
  dans la branche, jamais au niveau module — Mac Intel sans AVX2) ; `replay` : rejoue des
  fixtures JSON, zero appel modele.
- Sorties sales tolerees : extraction JSON robuste + enveloppe systematique des champs.

In [6]:
import base64, hashlib, json

def _images_vers_b64(images_bgr, qualite=92):
    b64s = []
    for img in images_bgr:
        if MAX_COTE_VLM and max(img.shape[:2]) > MAX_COTE_VLM:
            f = MAX_COTE_VLM / max(img.shape[:2])
            img = cv2.resize(img, None, fx=f, fy=f, interpolation=cv2.INTER_AREA)
        ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), qualite])
        if not ok:
            raise RuntimeError("Echec d'encodage JPEG")
        b64s.append(base64.b64encode(buf.tobytes()).decode())
    return b64s

def _cle_fixture(prompt, images_b64):
    h = hashlib.sha1()
    h.update(prompt.encode())
    for b in images_b64:
        h.update(hashlib.sha1(b.encode()).digest())
    return h.hexdigest()

_MODELE_DOMINO = {}

def _inference_ollama(images_b64, prompt, gen_kwargs):
    import requests
    corps = {"model": OLLAMA_MODEL,
             "messages": [{"role": "user", "content": prompt, "images": images_b64}],
             "stream": False,
             "options": {"temperature": 0, "num_predict": gen_kwargs.get("max_new_tokens", 1024)}}
    r = requests.post(f"{OLLAMA_URL}/api/chat", json=corps, timeout=600)
    r.raise_for_status()
    return r.json()["message"]["content"]

def _inference_domino(images_b64, prompt, gen_kwargs):
    # imports gardes : jamais executes en local
    import torch
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    from PIL import Image
    import io
    if "modele" not in _MODELE_DOMINO:
        _MODELE_DOMINO["modele"] = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            MODEL_PATH, torch_dtype=torch.bfloat16, device_map="auto")
        _MODELE_DOMINO["proc"] = AutoProcessor.from_pretrained(MODEL_PATH)
    modele, proc = _MODELE_DOMINO["modele"], _MODELE_DOMINO["proc"]
    pils = [Image.open(io.BytesIO(base64.b64decode(b))).convert("RGB") for b in images_b64]
    contenu = [{"type": "image", "image": im} for im in pils] + [{"type": "text", "text": prompt}]
    messages = [{"role": "user", "content": contenu}]
    texte = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    entrees = proc(text=[texte], images=pils, return_tensors="pt").to(modele.device)
    with torch.no_grad():
        sortie = modele.generate(**entrees, max_new_tokens=gen_kwargs.get("max_new_tokens", 1024),
                                 do_sample=False)
    genere = sortie[0][entrees["input_ids"].shape[1]:]
    return proc.decode(genere, skip_special_tokens=True)

def _inference_replay(images_b64, prompt, gen_kwargs):
    cle = _cle_fixture(prompt, images_b64)
    chemin = FIXTURES_DIR / f"{cle}.json"
    if not chemin.exists():
        raise FileNotFoundError(f"Fixture replay absente : {chemin} (enregistrer via RECORD_FIXTURES)")
    return json.loads(chemin.read_text())["reponse"]

def _run_inference(images_bgr, prompt, gen_kwargs=None):
    # Point d'entree unique du VLM. Les autres modules ignorent quel backend tourne.
    gen_kwargs = gen_kwargs or {}
    images_b64 = _images_vers_b64(images_bgr)
    if BACKEND == "replay":
        return _inference_replay(images_b64, prompt, gen_kwargs)
    if BACKEND == "ollama":
        reponse = _inference_ollama(images_b64, prompt, gen_kwargs)
    elif BACKEND == "domino":
        reponse = _inference_domino(images_b64, prompt, gen_kwargs)
    elif BACKEND == "openrouter":
        reponse = _inference_openrouter(images_b64, prompt, gen_kwargs)
    else:
        raise ValueError(f"BACKEND inconnu : {BACKEND}")
    if RECORD_FIXTURES:
        cle = _cle_fixture(prompt, images_b64)
        (FIXTURES_DIR / f"{cle}.json").write_text(
            json.dumps({"prompt": prompt, "reponse": reponse}, ensure_ascii=False, indent=2))
    return reponse

def _inference_openrouter(images_b64, prompt, gen_kwargs):
    import os, requests as _req
    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise ValueError("OPENROUTER_API_KEY absente de l'environnement")
    content = [{"type": "text", "text": prompt}]
    for b64 in images_b64:
        content.append({"type": "image_url",
                         "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})
    r = _req.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}",
                 "Content-Type": "application/json"},
        json={"model": OPENROUTER_MODEL,
              "messages": [{"role": "user", "content": content}],
              "max_tokens": gen_kwargs.get("max_new_tokens", 1024),
              "temperature": 0},
        timeout=120
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

def extraire_json(texte):
    # Extraction JSON robuste : retire les clotures ``` et isole la premiere structure {...}
    if texte is None:
        return None
    t = texte.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.MULTILINE).strip()
    debut = t.find("{")
    fin = t.rfind("}")
    if debut == -1 or fin <= debut:
        return None
    fragment = t[debut:fin + 1]
    fragment = re.sub(r",\s*([}\]])", r"\1", fragment)   # virgules terminales
    try:
        return json.loads(fragment)
    except json.JSONDecodeError:
        return None

def envelopper_champ(valeur):
    # Tout champ devient {"valeur","texte_brut","confiance"}, meme si le modele a repondu nu.
    if isinstance(valeur, dict):
        return {"valeur": valeur.get("valeur"),
                "texte_brut": valeur.get("texte_brut"),
                "confiance": valeur.get("confiance")}
    if valeur is None or (isinstance(valeur, str) and valeur.strip().lower() in ("", "null", "none")):
        return {"valeur": None, "texte_brut": None, "confiance": None}
    return {"valeur": valeur, "texte_brut": str(valeur), "confiance": None}

## Transcription MRZ (VLM) + analyse deterministe
Le VLM **recopie** la MRZ caractere par caractere (`<` compris). Le code seul decide :
- structure 3 lignes de ~30 -> TD1 ; 2 lignes de ~44 -> TD3 ; sinon invalide ;
- TD1/TD3 : pays emetteur = ligne 1, caracteres 3-5 (le discriminant DZ/etranger) ;
  la nationalite (TD3 ligne 2, car. 11-13) sert de recoupement ;
- sommes de controle ISO 9303 (poids 7-3-1) -> confiance chiffree, jamais ressentie ;
- desaccord avec l'hypothese heuristique -> l'analyse de structure fait autorite.

In [7]:
PROMPT_MRZ = (
    "Cette image contient une zone MRZ (lignes de caracteres avec des chevrons '<' en police "
    "monospace, en bas ou au dos du document). Recopie-la EXACTEMENT, caractere par caractere, "
    "en conservant tous les chevrons '<'. N'invente rien : si un caractere est illisible, mets '?'. "
    "Reponds UNIQUEMENT en JSON : {\"lignes\": [\"...\", \"...\"], \"confiance\": 0.0}"
)

_VALEURS_MRZ = {c: i for i, c in enumerate("0123456789")}
_VALEURS_MRZ.update({c: i + 10 for i, c in enumerate("ABCDEFGHIJKLMNOPQRSTUVWXYZ")})
_VALEURS_MRZ["<"] = 0

def _somme_controle(champ):
    poids = (7, 3, 1)
    try:
        total = sum(_VALEURS_MRZ[c] * poids[i % 3] for i, c in enumerate(champ))
    except KeyError:
        return None
    return total % 10

def _controle_ok(champ, chiffre):
    attendu = _somme_controle(champ)
    return attendu is not None and chiffre.isdigit() and attendu == int(chiffre)

def _date_mrz(aammjj, contexte):
    # AAMMJJ -> ISO. Le pivot de siecle est deterministe : naissance <= annee courante, expiration libre.
    import datetime
    if not re.fullmatch(r"\d{6}", aammjj):
        return None
    aa, mm, jj = int(aammjj[:2]), int(aammjj[2:4]), int(aammjj[4:6])
    annee_courante = datetime.date.today().year % 100
    siecle = 1900 if (contexte == "naissance" and aa > annee_courante) else 2000
    try:
        return datetime.date(siecle + aa, mm, jj).isoformat()
    except ValueError:
        return None

def parser_mrz(lignes):
    # Analyse deterministe. Retourne {"format","donnees","controles","valide"} ou None.
    if not lignes:
        return None
    lignes = [re.sub(r"\s", "", l).upper() for l in lignes if l and l.strip()]
    longueurs = [len(l) for l in lignes]
    if len(lignes) == 3 and all(27 <= n <= 33 for n in longueurs):
        fmt, cible = "TD1", 30
    elif len(lignes) == 2 and all(40 <= n <= 46 for n in longueurs):
        fmt, cible = "TD3", 44
    else:
        return None
    lignes = [l[:cible].ljust(cible, "<") for l in lignes]
    if fmt == "TD3":
        l1, l2 = lignes
        donnees = {"code_document": l1[0:2], "pays_emetteur": l1[2:5],
                   "numero": l2[0:9].replace("<", ""), "nationalite": l2[10:13],
                   "date_naissance": _date_mrz(l2[13:19], "naissance"), "sexe": l2[20],
                   "date_expiration": _date_mrz(l2[21:27], "expiration"),
                   "nom_mrz": l1[5:].split("<<")[0].replace("<", " ").strip(),
                   "prenoms_mrz": " ".join(p for p in l1[5:].split("<<")[1:] if p).replace("<", " ").strip()}
        controles = {"numero": _controle_ok(l2[0:9], l2[9]),
                     "date_naissance": _controle_ok(l2[13:19], l2[19]),
                     "date_expiration": _controle_ok(l2[21:27], l2[27])}
    else:
        l1, l2, l3 = lignes
        donnees = {"code_document": l1[0:2], "pays_emetteur": l1[2:5],
                   "numero": l1[5:14].replace("<", ""), "nationalite": l2[15:18],
                   "date_naissance": _date_mrz(l2[0:6], "naissance"), "sexe": l2[7],
                   "date_expiration": _date_mrz(l2[8:14], "expiration"),
                   "nom_mrz": l3.split("<<")[0].replace("<", " ").strip(),
                   "prenoms_mrz": " ".join(p for p in l3.split("<<")[1:] if p).replace("<", " ").strip()}
        controles = {"numero": _controle_ok(l1[5:14], l1[14]),
                     "date_naissance": _controle_ok(l2[0:6], l2[6]),
                     "date_expiration": _controle_ok(l2[8:14], l2[14])}
    nb_ok = sum(1 for v in controles.values() if v)
    return {"format": fmt, "donnees": donnees, "controles": controles,
            "valide": nb_ok >= 2, "nb_controles_ok": nb_ok}

def transcrire_mrz(img_page_redressee, boite=None, echelle=None):
    # Envoie au VLM la region MRZ (avec marge) si connue, sinon la page entiere.
    img = img_page_redressee
    if boite is not None and echelle:
        x1, y1, x2, y2 = [int(v / echelle) for v in boite]
        h, w = img.shape[:2]
        mx, my = int(0.05 * w), int(0.05 * h)
        img = img[max(0, y1 - my):min(h, y2 + my), max(0, x1 - mx):min(w, x2 + mx)]
    reponse = _run_inference([img], PROMPT_MRZ, {"max_new_tokens": 300})
    obj = extraire_json(reponse)
    if not obj or not isinstance(obj.get("lignes"), list):
        return None, reponse
    return obj, reponse

## Classification en 2 temps (decision = table deterministe)
1. Heuristique MRZ -> hypothese.
2. Si MRZ detectee : transcription VLM puis **la structure transcrite decide** (TD1 -> famille via
   `DOC_CODE_MAP` ; TD3 + emetteur `DZA` -> `PASSEPORT_DZ`, sinon `PASSEPORT_ETRANGER`).
3. Sans MRZ exploitable : classification VLM **fermee** parmi les 7 codes, confiance chiffree.
4. Desaccord ou confiance < `CLASSIF_CONF_MIN` -> `INCONNU` systematique, motif trace.

In [8]:
PROMPT_CLASSIF = (
    "Tu vois les pages d'un document d'identite scanne dans le cadre d'un dossier KYC. "
    "Le document peut etre algerien OU etranger. "
    "Choisis UN SEUL code parmi :\n"
    "- CNI_BIO : carte nationale d'identite biometrique (format carte, puce electronique, MRZ au dos)\n"
    "- CNI_NONBIO : carte nationale ancienne (carte ou livret, pas de puce, pas de MRZ)\n"
    "- PERMIS_BIO : permis de conduire biometrique (format carte, puce, tableau de categories)\n"
    "- PERMIS_NONBIO : permis livret papier 3 volets, tampons manuscrits, pas de MRZ\n"
    "- PASSEPORT_DZ : passeport algerien (pays emetteur DZA dans la MRZ)\n"
    "- PASSEPORT_ETRANGER : passeport etranger, tout pays hors Algerie\n"
    "- INCONNU : illisible, tronque, ou aucun des types ci-dessus\n"
    "Ne devine jamais : en cas de doute reponds INCONNU.\n"
    "Reponds UNIQUEMENT en JSON : {\"type\": \"CODE\", \"confiance\": 0.0}"
)

def _decision_depuis_mrz(analyse):
    donnees = analyse["donnees"]
    if analyse["format"] == "TD3":
        t = "PASSEPORT_DZ" if donnees["pays_emetteur"] == "DZA" else "PASSEPORT_ETRANGER"
        return t, "mrz_td3"
    famille = DOC_CODE_MAP.get(donnees["code_document"])
    if famille is None and donnees["code_document"].startswith("I"):
        famille = "CNI_BIO"
    if famille is None:
        return "INCONNU", "code_document_mrz_inconnu"
    return famille, "mrz_td1"

def classifier_document(pages_redressees, resultat_mrz):
    # Retourne {"type","confiance","methode","details"} — jamais de texte libre.
    details = {"heuristique_nb_lignes": resultat_mrz["nb_lignes"] if resultat_mrz else 0}

    # -- voie MRZ : transcription puis decision par structure --
    if resultat_mrz and resultat_mrz["nb_lignes"] >= 2 and resultat_mrz.get("page") is not None:
        obj, brut = transcrire_mrz(resultat_mrz["page"], resultat_mrz["boite"], resultat_mrz["echelle"])
        details["mrz_brute"] = obj["lignes"] if obj else None
        # retry page entiere si la transcription bbox n'a produit aucune ligne
        if not (obj and obj.get("lignes")):
            obj, _ = transcrire_mrz(resultat_mrz["page"])
            details["mrz_brute"] = obj["lignes"] if obj else None
            if obj and obj.get("lignes"):
                details["note_mrz_retry"] = "retry page entiere (bbox vide)"
        analyse = parser_mrz(obj["lignes"]) if obj else None
        if analyse:
            details["mrz_analyse"] = analyse
            if analyse["format"] == "TD1" and resultat_mrz["nb_lignes"] == 2:
                details["note"] = "heuristique 2 lignes corrigee par structure TD1"
            if analyse["valide"]:
                t, methode = _decision_depuis_mrz(analyse)
                conf = 0.7 + 0.1 * analyse["nb_controles_ok"]
                return {"type": t, "confiance": round(conf, 2), "methode": methode, "details": details}
            details["note_mrz"] = "controles ISO9303 insuffisants -> fallback VLM"

    # -- fallback : classification VLM fermee --
    reponse = _run_inference(pages_redressees[:2], PROMPT_CLASSIF, {"max_new_tokens": 100})
    obj = extraire_json(reponse) or {}
    t = obj.get("type")
    conf = obj.get("confiance")
    details["vlm_brut"] = reponse[:200]
    if t not in TYPES_PIECES or not isinstance(conf, (int, float)) or conf < CLASSIF_CONF_MIN:
        return {"type": "INCONNU", "confiance": float(conf or 0.0), "methode": "vlm_sous_seuil",
                "details": details}
    # garde-fou : PASSEPORT_DZ sans MRZ = trop risque ; PASSEPORT_ETRANGER autorise si conf >= seuil
    if t == "PASSEPORT_DZ" and "mrz_analyse" not in details:
        return {"type": "INCONNU", "confiance": float(conf), "methode": "passeport_dz_sans_mrz_valide",
                "details": details}
    return {"type": t, "confiance": float(conf), "methode": "vlm", "details": details}

## Schemas de champs par type + extraction
Un schema statique par type. Chaque champ extrait = `{"valeur","texte_brut","confiance"}`
(`texte_brut` : copie exacte, arabe preserve). Illisible -> `valeur: null` + `champs_illisibles`.
Pour un passeport etranger, l'absence de NIN algerien est **attendue** (`NA`), jamais un echec.

In [9]:
_CHAMPS_CARTE_BIO = ["nom", "prenom", "nom_arabe", "prenom_arabe", "date_naissance",
                     "lieu_naissance", "sexe", "nin", "numero_document",
                     "date_delivrance", "date_expiration", "autorite"]

SCHEMA_PAR_TYPE = {
    "CNI_BIO":       _CHAMPS_CARTE_BIO,
    "PERMIS_BIO":    _CHAMPS_CARTE_BIO + ["categories"],
    "CNI_NONBIO":    ["nom", "prenom", "nom_arabe", "prenom_arabe", "date_naissance",
                      "lieu_naissance", "numero_document", "date_delivrance", "autorite"],
    "PERMIS_NONBIO": ["nom", "prenom", "nom_arabe", "prenom_arabe", "date_naissance",
                      "lieu_naissance", "numero_document", "date_delivrance",
                      "categories", "wilaya"],
    "PASSEPORT_DZ":  ["type_document", "code_pays", "numero_document", "nom", "prenoms",
                      "nationalite", "date_naissance", "lieu_naissance", "sexe",
                      "date_delivrance", "date_expiration", "autorite", "nin"],
    "PASSEPORT_ETRANGER": ["type_document", "code_pays", "numero_document", "nom", "prenoms",
                           "nationalite", "date_naissance", "lieu_naissance", "sexe",
                           "date_delivrance", "date_expiration", "autorite"],
}

def _prompt_extraction(type_piece):
    champs = SCHEMA_PAR_TYPE[type_piece]
    liste = "\n".join(f"- {c}" for c in champs)
    return (
        f"Document : {type_piece}. Transcris UNIQUEMENT ce qui est visible sur les images.\n"
        f"Champs a extraire :\n{liste}\n\n"
        "Pour CHAQUE champ, reponds avec un objet {\"valeur\", \"texte_brut\", \"confiance\"} :\n"
        "- texte_brut : copie EXACTE de l'imprime (garde l'ecriture arabe d'origine telle quelle) ;\n"
        "- valeur : reformatage normalise (dates en AAAA-MM-JJ, texte latin en MAJUSCULES) ;\n"
        "- confiance : 0.0 a 1.0.\n"
        "Champ illisible ou absent : valeur=null et ajoute son nom dans champs_illisibles.\n"
        "N'INVENTE JAMAIS une valeur.\n"
        "Reponds UNIQUEMENT en JSON : {\"champs\": {nom_du_champ: {...}}, \"champs_illisibles\": []}"
    )

def extraire_champs(type_piece, pages_redressees):
    if type_piece not in SCHEMA_PAR_TYPE:
        return {"champs": {}, "champs_illisibles": [], "erreur": f"pas de schema pour {type_piece}"}
    reponse = _run_inference(pages_redressees, _prompt_extraction(type_piece),
                             {"max_new_tokens": 1600})
    obj = extraire_json(reponse) or {}
    bruts = obj.get("champs") or {}
    champs, illisibles = {}, list(obj.get("champs_illisibles") or [])
    for nom in SCHEMA_PAR_TYPE[type_piece]:
        champs[nom] = envelopper_champ(bruts.get(nom))
        if champs[nom]["valeur"] is None and nom not in illisibles:
            illisibles.append(nom)
    return {"champs": champs, "champs_illisibles": sorted(set(illisibles)),
            "reponse_brute": reponse[:500]}

## Orchestration : un dossier -> un JSON
Pour chaque `Identité*.pdf` : rendu -> triage blanches -> MRZ (heuristique + balayage) ->
redressement -> classification -> extraction. Les pages vides sont tracees, jamais envoyees au VLM.

In [10]:
def traiter_piece(chemin_pdf: Path):
    resultat = {"fichier": str(chemin_pdf), "pages": [], "classification": None, "extraction": None}
    pages = rendre_pdf(chemin_pdf)
    pages_utiles, meilleur_mrz = [], None
    for p in pages:
        blanche, mesures = est_page_blanche(p["image"])
        info = {"numero": p["numero"], "blanche": blanche,
                "mesures": {k: round(v, 4) for k, v in mesures.items()}}
        if not blanche:
            det = detecter_mrz(p["image"])
            info["mrz"] = {"nb_lignes": det["nb_lignes"], "orientation": det["orientation"],
                           "angle_fin": det["angle_fin"]}
            page_redressee = redresser_page(p["image"], det["orientation"], det["angle_fin"])
            pages_utiles.append(page_redressee)
            if det["nb_lignes"] > (meilleur_mrz or {"nb_lignes": 0})["nb_lignes"]:
                meilleur_mrz = dict(det, page=page_redressee)
        resultat["pages"].append(info)

    if not pages_utiles:
        resultat["classification"] = {"type": "INCONNU", "confiance": 0.0,
                                      "methode": "aucune_page_exploitable", "details": {}}
        return resultat

    resultat["classification"] = classifier_document(pages_utiles, meilleur_mrz)
    t = resultat["classification"]["type"]
    if t != "INCONNU":
        resultat["extraction"] = extraire_champs(t, pages_utiles)
    return resultat

def executer(data_root=DATA_ROOT, ids_referentiel=None):
    index, anomalies = localiser_dossiers(data_root, ids_referentiel)
    (OUT_DIR / "anomalies_localisation.json").write_text(
        json.dumps(anomalies, ensure_ascii=False, indent=2))
    recap = []
    for idc, contenu in index.items():
        for pdf in contenu["identite"]:
            print(f"[{idc}] {pdf.name} ...", flush=True)
            try:
                res = traiter_piece(pdf)
            except Exception as exc:   # une piece en echec ne tue jamais le run
                res = {"fichier": str(pdf), "erreur": repr(exc)}
            dossier_sortie = OUT_DIR / idc
            dossier_sortie.mkdir(parents=True, exist_ok=True)
            nom_json = (
                f"{idc}_"
                + re.sub(r"[^a-z0-9]+", "_", sans_accents(pdf.stem)).strip("_")
                + ".json"
            )
            (dossier_sortie / nom_json).write_text(
                json.dumps(res, ensure_ascii=False, indent=2, default=str))
            classif = res.get("classification") or {}
            recap.append({"id_tiers": idc, "fichier": pdf.name,
                          "type": classif.get("type"), "confiance": classif.get("confiance"),
                          "methode": classif.get("methode"),
                          "pages_blanches": sum(1 for pg in res.get("pages", []) if pg.get("blanche"))})
    return recap, anomalies

# --- lancement ---
import pandas as pd
ref = pd.read_excel("data_test/KYC /Donnees_Tiers_Extrait.xlsx", dtype=str)
recap, anomalies = executer(DATA_ROOT, ids_referentiel=ref["Id tiers"])
pd.DataFrame(recap)

[07000122810] Identité 13.pdf ...


[07000123041] identite 5.pdf ...


[07000123194] Identité 6.pdf ...


[07000123213] Identité 7.pdf ...


[07000123530] Identité 8.pdf ...


...

[07000124386] Identité 12.pdf ...


[07000124613] Identité 1.pdf ...


[07000125166] Identité 2.pdf ...


[07000125307] Identité 3.pdf ...


,id_tiers,fichier,type,confiance,methode,pages_blanches
0,07000122810,Identité 13.pdf,PERMIS_NONBIO,0.9,vlm,2
1,07000123041,identite 5.pdf,PERMIS_BIO,0.9,vlm,0
2,07000123194,Identité 6.pdf,CNI_NONBIO,0.9,vlm,0
3,07000123213,Identité 7.pdf,PASSEPORT_ETRANGER,1.0,vlm,0
4,07000123530,Identité 8.pdf,CNI_BIO,0.9,vlm,0
5,07000124069,Identité 11.pdf,PASSEPORT_ETRANGER,1.0,mrz_td3,0
6,07000124386,Identité 12.pdf,CNI_BIO,0.9,vlm,0
7,07000124613,Identité 1.pdf,PERMIS_NONBIO,0.9,vlm,0
8,07000125166,Identité 2.pdf,PERMIS_NONBIO,0.9,vlm,0
9,07000125307,Identité 3.pdf,PASSEPORT_ETRANGER,1.0,mrz_td3,0


## Limites connues (calibrees le 03/07/2026) et suite
- Heuristique MRZ : 15/18 pages sur specimens, 0 faux positif. Echecs -> fallback VLM (nominal).
  Cas 4/3 : 2 lignes detectees sur une TD1 -> corrige par la structure transcrite (garde-fou actif).
- Detection de puce non implementee (peu fiable sur photocopies N&B) : la classification repose
  sur MRZ + VLM ferme. A reevaluer si des scans couleur arrivent en production.
- `DOC_CODE_MAP` : verifier le code document du permis biometrique algerien sur specimen reel.
- `CLASSIF_CONF_MIN = 0.75` : a calibrer en local (courbe confiance/erreur sur les 13 specimens).
- Etapes suivantes (notebooks separes) : `identity_matcher`, `field_comparator`,
  `validity_checker`, rapport Excel.

In [ ]:
# === EXPORT EXCEL =============================================================
# Genere outputs/rapport_kyc.xlsx (toutes les lignes du referentiel + resultats pipeline).
# Feuille 1 : Resultats — une ligne par piece traitee. Feuille 2 : Anomalies.
# Executer APRES le lancement (cellule precedente).
# ==============================================================================

import json, unicodedata, re
from pathlib import Path
import pandas as pd
import openpyxl
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

# -- helpers concordance -------------------------------------------------------
def _normaliser_nom(s):
    """Minuscules, sans accents, sans ponctuation -> tokens comparables."""
    if not s:
        return ""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def _normaliser_date(s):
    """Extrait AAAA-MM-JJ depuis une chaine quelconque (ISO, DD/MM/YYYY, etc.)."""
    if not s:
        return None
    s = str(s).strip()
    m = re.match(r"(\d{4})-(\d{2})-(\d{2})", s)
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    m = re.match(r"(\d{2})[/\-](\d{2})[/\-](\d{4})", s)
    if m:
        return f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
    return None

def _concordance(raison_sociale, nom_vlm, prenom_vlm):
    """
    Compare Raison sociale du referentiel avec nom+prenom VLM.
    Retourne 'OK' si tous les tokens non vides du VLM sont dans la raison sociale,
    'A_VERIFIER' sinon, 'NA' si aucun nom VLM disponible.
    """
    ref    = _normaliser_nom(raison_sociale)
    tokens = [t for t in (_normaliser_nom(nom_vlm or "") + " " + _normaliser_nom(prenom_vlm or "")).split() if t]
    if not tokens:
        return "NA"
    return "OK" if all(t in ref for t in tokens) else "A_VERIFIER"

def _concordance_naissance(date_ref, lieu_ref, date_vlm, lieu_vlm):
    """
    'OK'        : date ET lieu presents et concordants.
    'A_VERIFIER': au moins un present mais discordant.
    'NA'        : date ou lieu VLM absent (champ non extrait).
    La date ref absente ou le lieu ref absent ne penalisent pas.
    Le lieu est compare par inclusion partielle (insensible casse/accents).
    """
    d_vlm = _normaliser_date(date_vlm)
    l_vlm = _normaliser_nom(lieu_vlm or "")
    if not d_vlm or not l_vlm:
        return "NA"
    d_ref = _normaliser_date(date_ref)
    l_ref = _normaliser_nom(lieu_ref or "")
    date_ok = (d_ref == d_vlm) if d_ref else True
    lieu_ok = (l_ref in l_vlm or l_vlm in l_ref) if l_ref else True
    return "OK" if (date_ok and lieu_ok) else "A_VERIFIER"

# -- chargement referentiel ----------------------------------------------------
ref_df = pd.read_excel("data_test/KYC /Donnees_Tiers_Extrait.xlsx", dtype=str).fillna("")
ref_df["_idc"] = ref_df["Id tiers"].apply(compacter)

# -- helpers extraction JSON ---------------------------------------------------
def _charger_jsons_dossier(idc):
    """Retourne [(res_dict, err_str|None)] — une entree par JSON trouve dans le dossier."""
    dossier = OUT_DIR / idc
    if not dossier.exists():
        return [(None, "dossier_sortie_absent")]
    jsons = sorted(dossier.glob("*.json"))
    if not jsons:
        return [(None, "aucun_json")]
    return [(json.loads(j.read_text()), None) for j in jsons]

def _champ_valeur(extraction, nom_champ):
    if not extraction:
        return None
    obj = (extraction.get("champs") or {}).get(nom_champ)
    return obj.get("valeur") if obj else None

def _passeport_robustesse(classif):
    if not classif or classif.get("type") != "PASSEPORT_ETRANGER":
        return ""
    details = classif.get("details") or {}
    if classif.get("methode") == "mrz_td3" or details.get("mrz_analyse"):
        return "mrz_td3"
    return "vlm_visuel"

# -- construction des lignes ---------------------------------------------------
anomalies_json = json.loads((OUT_DIR / "anomalies_localisation.json").read_text())
anom_index = {}
for a in anomalies_json:
    idc = compacter(a.get("id_tiers") or "")
    if idc:
        anom_index.setdefault(idc, []).append(a.get("motif", ""))

lignes = []
for _, row in ref_df.iterrows():
    idc = row["_idc"]
    for res, err in _charger_jsons_dossier(idc):
        classif    = (res or {}).get("classification") or {}
        extraction = (res or {}).get("extraction") or {}
        pages      = (res or {}).get("pages") or []

        # Nom du fichier source (extrait du champ "fichier" du JSON)
        fichier_json = (res or {}).get("fichier", "")
        nom_piece    = Path(fichier_json).name if fichier_json else ""

        type_detecte   = classif.get("type") or ("ERREUR" if err else "")
        methode        = classif.get("methode") or (err or "")
        confiance      = classif.get("confiance")
        pages_blanches = sum(1 for p in pages if p.get("blanche"))
        champs_ill     = ", ".join((extraction.get("champs_illisibles") or [])) if extraction else ""

        nom_vlm        = _champ_valeur(extraction, "nom")
        prenom_vlm     = _champ_valeur(extraction, "prenom") or _champ_valeur(extraction, "prenoms")
        date_naissance = _champ_valeur(extraction, "date_naissance")
        lieu_naissance = _champ_valeur(extraction, "lieu_naissance")
        numero_doc     = _champ_valeur(extraction, "numero_document")
        date_deliv     = _champ_valeur(extraction, "date_delivrance")
        date_expir     = _champ_valeur(extraction, "date_expiration")

        concordance       = _concordance(row.get("Raison sociale", ""), nom_vlm, prenom_vlm)
        concordance_naiss = _concordance_naissance(
            row.get("Date de naissance", ""), row.get("Lieu de naissance", ""),
            date_naissance, lieu_naissance)
        rob          = _passeport_robustesse(classif)
        anomalie_str = ", ".join(anom_index.get(idc, []))

        if (anomalie_str or type_detecte in ("INCONNU", "ERREUR") or err
                or concordance == "A_VERIFIER" or concordance_naiss == "A_VERIFIER"):
            statut = "RELECTURE_OBLIGATOIRE"
        elif rob == "vlm_visuel" or champs_ill:
            statut = "RELECTURE_PRIORITAIRE"
        else:
            statut = "AUTO_OK"

        lignes.append({
            "id_tiers":              row.get("Id tiers", ""),
            "fichier":               nom_piece,
            "raison_sociale":        row.get("Raison sociale", ""),
            "nom_vlm":               nom_vlm or "",
            "prenom_vlm":            prenom_vlm or "",
            "date_naissance":        date_naissance or "",
            "lieu_naissance":        lieu_naissance or "",
            "date_naissance_ref":    _normaliser_date(row.get("Date de naissance", "")) or "",
            "lieu_naissance_ref":    row.get("Lieu de naissance", ""),
            "concordance_naissance": concordance_naiss,
            "concordance_nom":       concordance,
            "type_ref":              row.get("Type de Document d'identification", ""),
            "type_detecte":          type_detecte,
            "methode":               methode,
            "confiance":             confiance,
            "passeport_robustesse":  rob,
            "numero_document":       numero_doc or "",
            "date_delivrance":       date_deliv or "",
            "date_expiration":       date_expir or "",
            "champs_illisibles":     champs_ill,
            "pages_blanches":        pages_blanches,
            "anomalie":              anomalie_str,
            "statut_relecture":      statut,
        })

# -- styles openpyxl -----------------------------------------------------------
FILL_ROUGE   = PatternFill("solid", fgColor="FFC7CE")
FILL_ORANGE  = PatternFill("solid", fgColor="FFD580")
FILL_VERT    = PatternFill("solid", fgColor="C6EFCE")
FILL_ENTETES = PatternFill("solid", fgColor="404040")
STATUT_FILL  = {
    "RELECTURE_OBLIGATOIRE": FILL_ROUGE,
    "RELECTURE_PRIORITAIRE": FILL_ORANGE,
    "AUTO_OK":               FILL_VERT,
}

entetes = {
    "id_tiers":              "Id tiers",
    "fichier":               "Fichier pièce",
    "raison_sociale":        "Raison sociale (réf.)",
    "nom_vlm":               "Nom (VLM)",
    "prenom_vlm":            "Prénom (VLM)",
    "date_naissance":        "Date naiss. (VLM)",
    "lieu_naissance":        "Lieu naiss. (VLM)",
    "date_naissance_ref":    "Date naiss. (réf.)",
    "lieu_naissance_ref":    "Lieu naiss. (réf.)",
    "concordance_naissance": "Concordance naiss.",
    "concordance_nom":       "Concordance nom",
    "type_ref":              "Type doc (réf.)",
    "type_detecte":          "Type détecté",
    "methode":               "Méthode",
    "confiance":             "Confiance",
    "passeport_robustesse":  "Robustesse passeport",
    "numero_document":       "N° document",
    "date_delivrance":       "Date délivrance",
    "date_expiration":       "Date expiration",
    "champs_illisibles":     "Champs illisibles",
    "pages_blanches":        "Pages blanches",
    "anomalie":              "Anomalie",
    "statut_relecture":      "Statut relecture",
}
largeurs = {
    "id_tiers": 14, "fichier": 26, "raison_sociale": 26, "nom_vlm": 16, "prenom_vlm": 20,
    "date_naissance": 14, "lieu_naissance": 18,
    "date_naissance_ref": 14, "lieu_naissance_ref": 18,
    "concordance_naissance": 18,
    "concordance_nom": 16, "type_ref": 22, "type_detecte": 18, "methode": 14,
    "confiance": 10, "passeport_robustesse": 20, "numero_document": 16,
    "date_delivrance": 15, "date_expiration": 15, "champs_illisibles": 32,
    "pages_blanches": 14, "anomalie": 24, "statut_relecture": 22,
}

# -- construction du classeur --------------------------------------------------
wb  = openpyxl.Workbook()
ws1 = wb.active
ws1.title = "Résultats"

colonnes = list(lignes[0].keys()) if lignes else []

for j, col in enumerate(colonnes, start=1):
    c = ws1.cell(row=1, column=j, value=entetes.get(col, col))
    c.fill = FILL_ENTETES
    c.font = Font(bold=True, color="FFFFFF")
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

for i, ligne in enumerate(lignes, start=2):
    fill = STATUT_FILL.get(ligne["statut_relecture"], FILL_VERT)
    for j, col in enumerate(colonnes, start=1):
        c = ws1.cell(row=i, column=j, value=ligne[col])
        c.fill = fill
        c.alignment = Alignment(vertical="center")

for j, col in enumerate(colonnes, start=1):
    ws1.column_dimensions[get_column_letter(j)].width = largeurs.get(col, 14)
ws1.row_dimensions[1].height = 30
ws1.freeze_panes = "E2"
ws1.auto_filter.ref = ws1.dimensions

# -- Feuille 2 : Anomalies -----------------------------------------------------
ws2 = wb.create_sheet("Anomalies")
for j, h in enumerate(["Motif", "Id tiers", "Chemin"], start=1):
    c = ws2.cell(row=1, column=j, value=h)
    c.fill = FILL_ENTETES
    c.font = Font(bold=True, color="FFFFFF")
    c.alignment = Alignment(horizontal="center")
for i, a in enumerate(anomalies_json, start=2):
    ws2.cell(row=i, column=1, value=a.get("motif", ""))
    ws2.cell(row=i, column=2, value=a.get("id_tiers", ""))
    ws2.cell(row=i, column=3, value=a.get("chemin", ""))
for j, w in enumerate([22, 16, 52], start=1):
    ws2.column_dimensions[get_column_letter(j)].width = w
ws2.auto_filter.ref = f"A1:C{max(len(anomalies_json) + 1, 2)}"

# -- sauvegarde ----------------------------------------------------------------
chemin_rapport = OUT_DIR / "rapport_kyc.xlsx"
wb.save(str(chemin_rapport))
nb_pieces = sum(1 for l in lignes if l["fichier"])
print(f"Rapport sauvegarde : {chemin_rapport}  ({len(lignes)} lignes, {nb_pieces} pieces traitees)")
